Progetto: "AI Ghostwriter" - Adattamento e Tuning 

Scenario La demo tecnica basata su Shakespeare è stata un successo. Ora però il cliente, una casa editrice italiana ("Dante & Co."), vuole utilizzare la tecnologia per generare testo in lingua italiana nello stile della Divina Commedia. 
Inoltre, il cliente si è lamentato che a volte il testo è "troppo caotico" e altre volte "troppo ripetitivo". Devi intervenire sul codice per permettere loro di calibrare la creatività dell'AI. 

Obiettivi 
    1. Data Swapping: Sostituire il dataset di Shakespeare con quello di Dante Alighieri. 
    2. Hyperparameter Tuning: Modificare la "Temperatura" per trovare il bilanciamento perfetto tra grammatica corretta e creatività poetica. 
    3. Prompt Engineering: Adattare il seme di generazione (start_string) al nuovo contesto. 
    
Istruzioni Operative 

1. Analisi del Prototipo (Shakespeare) 
Esegui il codice base fornito (quello su Shakespeare) così com'è. Assicurati che funzioni e genera una frase di test. 
    * Analisi: Osserva l'output della funzione elabora_dizionari . Quanti caratteri unici usava Shakespeare? 
    * Annota questo numero, ti servirà per il confronto. 
    
2. Cambio del Dataset (Dante Alighieri) 
Dobbiamo cambiare la fonte dei dati. Il metodo get_file di Keras a volte dà problemi con link esterni, quindi useremo la libreria standard requests .
    * Task: modifica la funzione di download (o creane una nuova scarica_testo_dante) utilizzando questo link specifiche che abbiamo verificato essere funzionante:
    https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt 
    * Encoding: L'italiano ha accenti (à, è, ì, ò, ù). Se non specifichi la codifica, vedrai simboli strani. Assicurati di impostare response.encoding = 'utf-8' . 
    
    ** Suggerimento Codice:** 
    import requests 
    import urllib3 
    
    # Disabilitiamo i messaggi di avviso "InsecureRequestWarning" per avere un output pulito 
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) 
    
    def scarica_testo_dante():     
        """     Scarica la Divina Commedia ignorando gli errori di certificato SSL.     """     
        url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"     
        print(f"Scaricamento testo da: {url}...")     
        try:         
            # AGGIUNTA FONDAMENTALE: verify=False         
            # Dice a Python: "Fidati del server anche se non riconosci il certificato        "         
            response = requests.get(url, verify=False)         
            response.raise_for_status()         
            response.encoding = 'utf-8'          
            text = response.text         
            print(f"Download completato. Lunghezza testo: {len(text)} caratteri")         
            print(f"Esempio (primi 100 caratteri):\n{text[:100]}")         
            return text     
        except Exception as e:         
            print(f"Errore ancora presente: {e}") 
            # Se fallisce anche così, usiamo un link alternativo di GitHub (spesso più compatibile)
            print("Tentativo con link di riserva (GitHub)...")         
            try:             
                url_backup = "https://raw.githubusercontent.com/wpm/t-snetext-vis/master/data/divina_commedia.txt"             
                r2 = requests.get(url_backup, verify=False)             
                r2.encoding = 'utf-8'             
                return r2.text         
            except:             
                return None 
        # Eseguiamo il download 
        text = scarica_testo_dante() 

3. Adattamento dei Parametri di Training 
Dante scrive in terzine ed è molto strutturato, diversamente dai monologhi inglesi. 
    * Modifica crea_dataset_addestramento : Le terzine di Dante sono più brevi. Prova a modificare seq_length (lunghezza sequenza). Invece di 100 caratteri, prova a scendere a 80 o 60. 
    * Riflessione: Come influisce questo cambiamento sulla velocità di ogni epoca? 
    
4. Training (Addestramento) 
    * Rilancia l'addestramento sul nuovo testo. 
    * Consiglio: Poiché stiamo cambiando lingua e struttura, monitora la loss . Se non scende abbastanza, potresti dover aumentare il numero di epochs (prova almeno 10-20 epoche per risultati leggibili). 

5. Generazione e Tuning della Temperatura (Il cuore dell'esercizio) 
Il cliente vuole controllare lo stile. Modifica la parte finale ( main e genera_testo ): 
    1. Cambia il Prompt: Non usare più "ROMEO: " . Usa qualcosa di dantesco, es: "Nel mezzo " , "Selva " o "Virgilio ". 
    2. Esperimento Temperatura: Genera testo con tre configurazioni diverse e osserva i risultati: 
        * Temp = 0.2 (Freddo/Conservativo): Il modello entra in loop ripetendo sempre la stessa parola o frase? 
        * Temp = 1.0 (Standard): La sintassi e la grammatica reggono?
        * Temp = 2.0 (Caldo/Folle): Ci sono parole inventate? Sembra una lingua aliena?

Al termine del notebook, aggiungi una cella di testo con le tue conclusioni: 
    1. Quale temperatura ha generato la "terzina" più credibile? 
    2. Incolla qui sotto il miglior testo generato dalla tua AI.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import requests
import numpy as np
import tensorflow as tf
import keras
import torch

print("Backend Keras:", keras.backend.backend())
print("CUDA PyTorch disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch sta usando la CPU")  


Backend Keras: torch
CUDA PyTorch disponibile: False
PyTorch sta usando la CPU


In [3]:
def scarica_testo_dante():
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scarico testo da: {url}...")

    try:
        response = requests.get(url, verify=False, timeout=30)
        response.raise_for_status()
        response.encoding = "utf-8"

        text = response.text

        print(
            f"Download completato. "
            f"Lunghezza testo: {len(text)} caratteri"
        )
        return text

    except Exception as e:
        print(f"Errore sul primo indirizzo: {e}")
        print("Tentativo con link di riserva...")

        try:
            url_backup = (
                "https://raw.githubusercontent.com/"
                "wpm/t-snetext-vis/master/data/"
                "divina_commedia.txt"
            )

            response = requests.get(
                url_backup,
                verify=False,
                timeout=30
            )
            response.raise_for_status()
            response.encoding = "utf-8"

            return response.text

        except Exception as backup_error:
            raise RuntimeError(
                "Impossibile scaricare il testo"
            ) from backup_error

In [4]:
text = scarica_testo_dante()
if not text:
    raise ValueError("Il testo scaricato è vuoto o None")
#print(text[:500])

Scarico testo da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dmf.unicatt.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Download completato. Lunghezza testo: 551846 caratteri


In [5]:
#costruisco vocabolario
vocab = sorted(set(text))
vocab_size=len(vocab)
print(f"Il testo contiene {vocab_size} caratteri unici.")
#mappatura caratteri
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)
#converto in numeri il testo
text_as_int = np.array([char2idx[c] for c in text])
print(f"Testo vettorizzato (primi 10): {text_as_int[:10]}")


Il testo contiene 69 caratteri unici.
Testo vettorizzato (primi 10): [23 14  2 17 22 33 22 25 14  2]


In [6]:
# --- CREAZIONE DEL DATASET OTTIMIZZATO ---
seq_length = 80
BATCH_SIZE = 16
BUFFER_SIZE = 10000

char_dataset = tf.data.Dataset.from_tensor_slices(
    text_as_int
)

sequences = char_dataset.batch(
    seq_length + 1,
    drop_remainder=True
)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]

    return input_text, target_text

dataset = sequences.map(
    split_input_target,
    num_parallel_calls=tf.data.AUTOTUNE
)

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(
        BATCH_SIZE,
        drop_remainder=True
    )
    .prefetch(tf.data.AUTOTUNE)
)



In [7]:
#parametri

embedding_dim = 128
rnn_units = 256

MODELLO

In [8]:
# --- DEFINIZIONE DEL MODELLO (CON 3 LAYER GRU) ---
#tolgo 2 gru per pc poco performante
vocab_size = len(vocab)

def build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE):
    model = keras.Sequential([
        # Input
        keras.layers.Input(shape=(None,),batch_size=BATCH_SIZE,dtype="int64"),
        
        # Embedding Layer
        keras.layers.Embedding(vocab_size, embedding_dim),
        
        # 1° Layer GRU
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False, recurrent_initializer='glorot_uniform'),
        
        # 2° Layer GRU (Intermedio - Modifica richiesta)
        #keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        
        # 3° Layer GRU (Aggiunto per aumentare la capacità di apprendimento)
        #keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        
        # Dense Layer (Logits)
        keras.layers.Dense(vocab_size)
    ])
    return model

In [ ]:
model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)
# --- TRAINING ---
model.compile(optimizer="adam",loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

model.summary()
# Nota: Epochs impostate a 50 come richiesto; ridurre per test rapidi

epochs=100
#per gestire le epoche
early_stopping = keras.callbacks.EarlyStopping(
    monitor="loss", #controlla la loss
    mode="min",
    patience=5, #aspetta 5 epoche senza miglioramenti
    min_delta=0.001,
    restore_best_weights=True #per ritornare ai pesi del modello migliore (nonotante la patience)
)
#per gestire il learning rate
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_delta=0.001,
    min_lr=1e-6,
    verbose=1
)
#per salvare il modello migliore fino a quel momento
checkpoint = keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="loss",
    mode="min",
    save_best_only=True
)

history = model.fit(
    dataset,
    epochs=epochs,
    callbacks=[
        reduce_lr,
        early_stopping,
        checkpoint
    ]
)

# Grazie a restore_best_weights=True,
# model contiene già i pesi dell'epoca migliore.
model.save("dante.keras")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (16, None, 128)        │         8,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (16, None, 256)        │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (16, None, 69)         │        17,733 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323,013 (1.23 MB)

 Trainable params: 323,013 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 224s 523ms/step - loss: 2.2758 - learning_rate: 0.0010
Epoch 2/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 172s 401ms/step - loss: 1.8155 - learning_rate: 0.0010
Epoch 3/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 240s 562ms/step - loss: 1.6780 - learning_rate: 0.0010
Epoch 4/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 188s 440ms/step - loss: 1.5965 - learning_rate: 0.0010
Epoch 5/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 158s 371ms/step - loss: 1.5428 - learning_rate: 0.0010
Epoch 6/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 186s 436ms/step - loss: 1.5034 - learning_rate: 0.0010
Epoch 7/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 183s 429ms/step - loss: 1.4736 - learning_rate: 0.0010
Epoch 8/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 186s 437ms/step - loss: 1.4475 - learning_rate: 0.0010
Epoch 9/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 195s 457ms/step - loss: 1.4259 - learning_rate: 0.0010
Epoch 10/100
425/425 ━━━━━━━━━━━━━━━━━━━━ 191s 448ms/step - loss: 1.4071 - learning_rate: 0.0010
Epoch 11/100
425/425 ━━━━━━━━━━━━━━━━━━

In [ ]:

#per vedere l'epoca migliore
validation_loss = history.history["val_loss"]

best_epoch = int(np.argmin(validation_loss)) + 1
best_val_loss = float(np.min(validation_loss))
best_train_loss = history.history["loss"][best_epoch - 1]

print(f"Epoca migliore: {best_epoch}")
print(f"Validation loss migliore:{best_val_loss:.6f}")
print(f"Training loss alla migliore epoca: {best_train_loss:.6f}")

Migliore epoca: 50
Migliore loss: 1.2699462175369263


In [ ]:
#per verificare il lr finale
print("Learning rate finale:",
    float(keras.ops.convert_to_numpy(model.optimizer.learning_rate))
)

Learning rate finale: 0.0010000000474974513


In [ ]:
#Analizzo l'addestramento
import matplotlib.pyplot as plt

epoche_eseguite = range(
    1,
    len(history.history["loss"]) + 1
)

plt.figure(figsize=(10, 5))

plt.plot(
    epoche_eseguite,
    history.history["loss"],
    label="Training loss"
)

plt.plot(
    epoche_eseguite,
    history.history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.title("Andamento della loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# --- FUNZIONE DI GENERAZIONE E ANALISI DELLA TEMPERATURA ---
def generate_text(model, start_string, temperature=0.7, num_generate=400):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    # Disattiviamo i gradienti per risparmiare memoria durante l'inferenza
    with torch.no_grad():
        for i in range(num_generate):
            predictions = model(input_eval)
            # Applichiamo la temperatura ai logits
            predictions = predictions[:, -1, :] / temperature
            
            predicted_id_tensor = keras.random.categorical(predictions, num_samples=1)
            predicted_id = int(keras.ops.convert_to_numpy(predicted_id_tensor)[0, 0])

            new_char_tensor = tf.expand_dims([predicted_id], 0)
            input_eval = tf.concat([input_eval, new_char_tensor], axis=-1)
            
            # Sliding window per mantenere il contesto
            if input_eval.shape[1] > seq_length:
                input_eval = input_eval[:, 1:]

            text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

# --- LOOP DI SPERIMENTAZIONE ---
temperatures = [0.2, 1.0, 2.0]
seed = "Nel mezzo"

print(f"\n{'='*50}")
print(f"ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: {seed})")
print(f"{'='*50}\n")

for temp in temperatures:
    print(f"--- TEST CON TEMPERATURA: {temp} ---")
    generated = generate_text(model, start_string=seed, temperature=temp, num_generate=400)
    print(generated)
    print(f"\n{'-'*50}\n")
   


ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: Nel mezzo : )

--- TEST CON TEMPERATURA: 0.1 ---
Nel mezzo : Canto XXII

  Poscia che la mia vista che si convene
senti' di là da l'altro che la mia vista,
come fa che si fece la mia vista,
come si scorgerà di costui di sotto
come l'uom che di là da l'altro con l'altro scheggio
che se non come si discerner per la spera
che se non convien che si convene
senti' cominciò a l'altro che si sconde
sovra 'l principio di là da l'altra parte in su la costa

--------------------------------------------------

--- TEST CON TEMPERATURA: 1.0 ---
Nel mezzo : Ondalsito m'avea, soccordicli
queste parole giù posse' in che si s'accosta;
  e nel mio reggendo incosuta e Volsi
l'abitate don'affatini
per la lore in arte nel proprio stretto al foglia t'assolate,
che le nove così fatto specolo?", fini, quasi tu d'altro, speccia,
possibil che midicami raggia
d'avero ancora ad ogne capo, e facieno
con le mie par ch'una s'abbascia,
  che tu non ti passi i

----------